In [ ]:
import os

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
os.environ["groq_api_key"] = os.getenv("groq_api_key")

In [ ]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="qwen-2.5-32b")

In [ ]:
llm.invoke("hii")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
file_path = r"C:\Users\ROHIT\OneDrive\Desktop\PN\RohitVyavahare_DataScientist(AI_Engineer)_Resume.pdf"
loader = PyPDFLoader(file_path)
docs=loader.load()
len(docs)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=50)
splits=text_splitter.split_documents(docs)

In [ ]:
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams


In [ ]:
client = QdrantClient(":memory:")

client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

In [ ]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="demo_collection",
    embedding=embeddings
)

In [ ]:
vector_store.add_documents(documents=splits)

In [ ]:
retreiver=vector_store.as_retriever()
retreiver.invoke("skills")

In [ ]:
results = vector_store.similarity_search(
    "skills", k=2
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = (
                "You are an assistant for question-answering tasks. "
                "Use the following pieces of retrieved context to answer "
                "the question. If you don't know the answer, say that you "
                "don't know. Use three sentences maximum and keep the "
                "answer concise."
                "{context}")

In [ ]:
prompt=ChatPromptTemplate.from_messages(
    [
    ("system", system_prompt),
    ("user" ,"{input}")
])

In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [ ]:
qa_chain= create_stuff_documents_chain(llm,prompt)
chain=create_retrieval_chain(retreiver,qa_chain)
result=chain.invoke({"input": "who is rohit"})
result["answer"]

In [ ]:
print(result["context"][0].metadata)

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain_core.messages import AIMessage, HumanMessage
from langchain.chains import ConversationalRetrievalChain

# Create memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create the conversational chain
chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retreiver,
    memory=memory,
    verbose=True
)

# Example usage with memory
result = chain.invoke({"question": "who is rohit?"})
print(result["answer"])

# Ask follow-up question
result = chain.invoke({"question": "what are his skills?"})
print(result["answer"])